# APIM ❤️ Microsoft Foundry

## End-to-End Private Lab
![architecture](images/foundry-e2e-private.gif)

This lab deploys an **end-to-end private** Microsoft Foundry environment fronted by `Azure API Management` (StandardV2) acting as the AI Gateway. All backend services (Foundry, AI Search, Cosmos DB, Storage, the cross-region OpenAI account, and APIM itself) sit behind private endpoints in a single VNet. A Windows jump-box VM accessible via `Azure Bastion` is included so you can test the private endpoints from inside the VNet.

### What gets deployed
- Microsoft Foundry account + project with `networkInjections` to a private agent subnet
- Standard backend resources (Azure AI Search, Cosmos DB, Storage) behind private endpoints with private DNS zones
- APIM `StandardV2` with outbound VNet integration **and** an inbound private endpoint
- An Azure OpenAI inference API imported into APIM, with a managed-identity inbound policy (see [policy.xml](policy.xml))
- An APIM gateway connection on the Foundry project so models can be called as `apim-gateway/<model-name>`
- A cross-region Azure OpenAI account with private endpoint into the primary VNet, exposed through APIM as `apim-gateway-crossregion/<model-name>`
- Application Insights + Log Analytics, connected to the Foundry project for agent tracing
- Azure Bastion + Windows jump-box VM

### Notes
- Provisioning APIM `StandardV2` with VNet integration takes **~30 minutes**.
- The Foundry project endpoint is private. Run the test cells from the jump-box VM (via Bastion) once the deployment is complete.
- See the [clean-up notebook](clean-up-resources.ipynb) when you are done.

### Prerequisites
- [Python 3.12 or later](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) with the [Jupyter extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter)
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and signed in
- An Azure subscription with `Contributor` permissions and quota for the chosen models in **both** the primary and cross-region locations

<a id='0'></a>
### 0️⃣ Initialize notebook variables

In [ ]:
import os; globals().setdefault('__vsc_ipynb_file__', os.path.abspath('foundry-e2e-private.ipynb'))
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = 'eastus2'
cross_region_location = 'swedencentral'

# Primary model exposed through apim-gateway/<model>
model_name = 'gpt-4o-mini'
model_version = '2024-07-18'
model_capacity = 30
model_sku = 'GlobalStandard'

# Cross-region model exposed through apim-gateway-crossregion/<model>
deploy_cross_region = True
cross_region_model_name = 'gpt-4o'
cross_region_model_version = '2024-11-20'

# APIM
apim_sku = 'StandardV2'
publisher_email = 'apim-admin@contoso.com'
publisher_name = 'AI Foundry'
apim_connection_name = 'apim-gateway'
apim_cross_region_connection_name = 'apim-gateway-crossregion'
inference_api_version = '2024-10-21'

# Bastion + jumpbox
deploy_bastion = True
jumpbox_admin_username = 'azureuser'
jumpbox_admin_password = '@Aa123456789'   # change me before sharing

deploy_application_insights = True

utils.print_ok('Notebook initialized')

<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

In [ ]:
output = utils.run('az account show', 'Retrieved az account', 'Failed to get the current az account')

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']
    utils.print_info(f'Current user: {current_user}')
    utils.print_info(f'Tenant ID:    {tenant_id}')
    utils.print_info(f'Subscription: {subscription_id}')

<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This deploys the full private topology. **Provisioning APIM StandardV2 with VNet integration takes ~30 minutes**, so the deployment as a whole typically completes in 35-45 minutes.

In [ ]:
utils.create_resource_group(resource_group_name, resource_group_location)

bicep_parameters = {
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'location':                       { 'value': resource_group_location },
        'locationCrossRegion':            { 'value': cross_region_location },
        'modelName':                      { 'value': model_name },
        'modelVersion':                   { 'value': model_version },
        'modelCapacity':                  { 'value': model_capacity },
        'modelSkuName':                   { 'value': model_sku },
        'apimSku':                        { 'value': apim_sku },
        'publisherEmail':                 { 'value': publisher_email },
        'publisherName':                  { 'value': publisher_name },
        'apimConnectionName':             { 'value': apim_connection_name },
        'inferenceApiVersion':            { 'value': inference_api_version },
        'deployCrossRegionOpenAI':        { 'value': deploy_cross_region },
        'crossRegionModelName':           { 'value': cross_region_model_name },
        'crossRegionModelVersion':        { 'value': cross_region_model_version },
        'apimCrossRegionConnectionName':  { 'value': apim_cross_region_connection_name },
        'deployApplicationInsights':      { 'value': deploy_application_insights },
        'deployBastion':                  { 'value': deploy_bastion },
        'jumpboxAdminUsername':           { 'value': jumpbox_admin_username },
        'jumpboxAdminPassword':           { 'value': jumpbox_admin_password },
    }
}

with open('params.json', 'w') as f:
    f.write(json.dumps(bicep_parameters))

output = utils.run(
    f'az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json',
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed")

<a id='3'></a>
### 3️⃣ Capture deployment outputs

In [ ]:
output = utils.run(
    f'az deployment group show --name {deployment_name} -g {resource_group_name}',
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}")

ai_project_endpoint = ''
apim_gateway_url = ''
apim_gateway_connection = ''
apim_cross_region_connection = ''
app_insights_connection_string = ''
jumpbox_name = ''
bastion_name = ''

if output.success and output.json_data:
    ai_services_name              = utils.get_deployment_output(output, 'aiServicesName',              'AI Services Name')
    ai_project_name               = utils.get_deployment_output(output, 'aiProjectName',               'AI Project Name')
    ai_project_endpoint           = utils.get_deployment_output(output, 'aiProjectEndpoint',           'AI Project Endpoint')
    apim_service_name             = utils.get_deployment_output(output, 'apimServiceName',             'APIM Service Name')
    apim_gateway_url              = utils.get_deployment_output(output, 'apimGatewayUrl',              'APIM Gateway URL')
    apim_gateway_connection       = utils.get_deployment_output(output, 'apimGatewayConnectionName',   'APIM Gateway Connection Name')
    apim_cross_region_connection  = utils.get_deployment_output(output, 'apimCrossRegionConnectionName', 'APIM Cross-Region Connection Name')
    app_insights_connection_string = utils.get_deployment_output(output, 'appInsightsConnectionString', 'App Insights Connection String', True)
    jumpbox_name                  = utils.get_deployment_output(output, 'jumpboxName',                 'Jumpbox VM Name')
    bastion_name                  = utils.get_deployment_output(output, 'bastionName',                 'Bastion Name')

<a id='4'></a>
### 4️⃣ 🧪 Test the agent through the APIM gateway

Because the Foundry project is **private**, the cells below need to run from a host that is inside (or peered with) the lab VNet — typically the jump-box VM created by this lab.

On the jump-box, install the dependencies and run something like:

```python
# pip install azure-ai-projects azure-identity
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

PROJECT_ENDPOINT = '<aiProjectEndpoint output>'
AGENT_MODEL      = 'apim-gateway/gpt-4o-mini'   # apim_connection_name + '/' + model_name

with AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential()) as project:
    agent = project.agents.create_agent(
        model=AGENT_MODEL,
        name='gateway-demo',
        instructions='You are a helpful assistant routed through the APIM AI Gateway.',
    )
    thread = project.agents.threads.create()
    project.agents.messages.create(thread_id=thread.id, role='user', content='Tell me one fun fact about Azure API Management.')
    run = project.agents.runs.create_and_process(thread_id=thread.id, agent_id=agent.id)
    print('Run status:', run.status)
    for msg in project.agents.messages.list(thread_id=thread.id):
        for c in msg.content:
            if hasattr(c, 'text'):
                print(f'{msg.role}: {c.text.value}')
    project.agents.delete_agent(agent.id)
```

All traffic flows: **agent → Data Proxy (private subnet) → APIM private endpoint → APIM → Foundry private endpoint**.

<a id='5'></a>
### 5️⃣ 🧪 Test the cross-region OpenAI through APIM

Same pattern as above, but use the cross-region connection name as the model prefix:

```python
AGENT_MODEL = 'apim-gateway-crossregion/gpt-4o'   # apim_cross_region_connection_name + '/' + cross_region_model_name
```

This routes the request through APIM, which forwards it (using its managed identity) to the OpenAI account deployed in the secondary region. The cross-region account is reachable from the primary VNet through its own private endpoint.

<a id='6'></a>
### 6️⃣ 🖥️ Connect to the jump-box via Azure Bastion

Run from your local machine to open an RDP-over-Bastion tunnel:

```bash
# get the resource IDs
RG=<resource_group_name>
VM_ID=$(az vm show -g $RG -n <jumpbox_name> --query id -o tsv)

# Native client RDP tunnel (Windows jump-box)
az network bastion rdp \
  --name <bastion_name> \
  --resource-group $RG \
  --target-resource-id $VM_ID
```

Or in the Azure Portal: navigate to the jump-box VM → **Connect** → **Bastion**, then sign in with `azureuser` and the password you set in step 0.

Once on the VM, install Python + the SDK and run the snippet from step 4.

<a id='clean'></a>
### 🗑️ Clean up resources

When you are finished with the lab, run the [clean-up notebook](clean-up-resources.ipynb) to delete the resource group and avoid extra charges.